# 🧠 Deep Learning: PyTorch LSTM Time Series Forecasting

## 📌 Objective
Build and train a Recurrent Neural Network using **PyTorch LSTM (Long Short-Term Memory)** to model temporal sequential dependencies in Aadhaar daily enrolment time-series across states.

### 🏗️ Architecture Design
- **Input Sequence Length (Lookback Window)**: 14 past days
- **Forecast Horizon**: 7 days ahead (or multi-step recursive projection)
- **Model Stack**:
  1. `nn.LSTM(input_size, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)`
  2. `nn.Linear(64, output_size)`
- **Optimization**: Adam Optimizer, MSE Loss with Min-Max Feature Scaling (fitted on Train set only).

In [1]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import plotly.graph_objects as go

# Inline Data Ingestion & State Normalization Utilities
import os
import glob
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

CANONICAL_STATES = [
    "Andaman & Nicobar", "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chandigarh", "Chhattisgarh", "Dadra & Nagar Haveli and Daman & Diu", "Delhi",
    "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Jharkhand",
    "Karnataka", "Kerala", "Ladakh", "Lakshadweep", "Madhya Pradesh", "Maharashtra",
    "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Puducherry", "Punjab",
    "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal"
]

STATE_ALIASES = {
    "andaman and nicobar islands": "Andaman & Nicobar",
    "andaman & nicobar islands": "Andaman & Nicobar",
    "a & n islands": "Andaman & Nicobar",
    "andhra pradesh": "Andhra Pradesh",
    "arunachal pradesh": "Arunachal Pradesh",
    "assam": "Assam",
    "bihar": "Bihar",
    "chandigarh": "Chandigarh",
    "chhattisgarh": "Chhattisgarh",
    "chhatisgarh": "Chhattisgarh",
    "dadra and nagar haveli": "Dadra & Nagar Haveli and Daman & Diu",
    "daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "dadra and nagar haveli and daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "delhi": "Delhi",
    "nct of delhi": "Delhi",
    "goa": "Goa",
    "gujarat": "Gujarat",
    "haryana": "Haryana",
    "himachal pradesh": "Himachal Pradesh",
    "jammu and kashmir": "Jammu and Kashmir",
    "jammu & kashmir": "Jammu and Kashmir",
    "jharkhand": "Jharkhand",
    "karnataka": "Karnataka",
    "kerala": "Kerala",
    "ladakh": "Ladakh",
    "lakshadweep": "Lakshadweep",
    "madhya pradesh": "Madhya Pradesh",
    "maharashtra": "Maharashtra",
    "manipur": "Manipur",
    "meghalaya": "Meghalaya",
    "mizoram": "Mizoram",
    "nagaland": "Nagaland",
    "odisha": "Odisha",
    "orissa": "Odisha",
    "puducherry": "Puducherry",
    "pondicherry": "Puducherry",
    "punjab": "Punjab",
    "rajasthan": "Rajasthan",
    "sikkim": "Sikkim",
    "tamil nadu": "Tamil Nadu",
    "telangana": "Telangana",
    "tripura": "Tripura",
    "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand",
    "uttaranchal": "Uttarakhand",
    "west bengal": "West Bengal"
}

def _normalize_state_name(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    return STATE_ALIASES.get(val_str, str(val).strip().title())

def load_and_preprocess_raw_data(data_dir="."):
    enrol_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_enrolment/**/*.csv'), recursive=True))
    enrol_list = []
    for f in enrol_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        enrol_list.append(df)
    
    enrol_df = pd.concat(enrol_list, ignore_index=True) if enrol_list else pd.DataFrame()
        
    if not enrol_df.empty:
        enrol_df['date'] = pd.to_datetime(enrol_df['date'], format='%d-%m-%Y', errors='coerce')
        enrol_df['norm_state'] = enrol_df['state'].apply(_normalize_state_name)
        enrol_df['total_enrolments'] = enrol_df['age_0_5'].fillna(0) + enrol_df['age_5_17'].fillna(0) + enrol_df['age_18_greater'].fillna(0)
        enrol_daily = enrol_df.groupby(['date', 'norm_state'])[['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments']].sum().reset_index()
    else:
        enrol_daily = pd.DataFrame(columns=['date', 'norm_state', 'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments'])

    demo_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_demographic/**/*.csv'), recursive=True))
    demo_list = []
    for f in demo_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        demo_list.append(df)
    
    if demo_list:
        demo_df = pd.concat(demo_list, ignore_index=True)
        demo_df['date'] = pd.to_datetime(demo_df['date'], format='%d-%m-%Y', errors='coerce')
        demo_df['norm_state'] = demo_df['state'].apply(_normalize_state_name)
        demo_df['demo_total'] = demo_df['demo_age_5_17'].fillna(0) + demo_df['demo_age_17_'].fillna(0)
        demo_daily = demo_df.groupby(['date', 'norm_state'])[['demo_age_5_17', 'demo_age_17_', 'demo_total']].sum().reset_index()
    else:
        demo_daily = pd.DataFrame(columns=['date', 'norm_state', 'demo_age_5_17', 'demo_age_17_', 'demo_total'])

    bio_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_biometric/**/*.csv'), recursive=True))
    bio_list = []
    for f in bio_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        bio_list.append(df)
        
    if bio_list:
        bio_df = pd.concat(bio_list, ignore_index=True)
        bio_df['date'] = pd.to_datetime(bio_df['date'], format='%d-%m-%Y', errors='coerce')
        bio_df['norm_state'] = bio_df['state'].apply(_normalize_state_name)
        bio_df['bio_total'] = bio_df['bio_age_5_17'].fillna(0) + bio_df['bio_age_17_'].fillna(0)
        bio_daily = bio_df.groupby(['date', 'norm_state'])[['bio_age_5_17', 'bio_age_17_', 'bio_total']].sum().reset_index()
    else:
        bio_daily = pd.DataFrame(columns=['date', 'norm_state', 'bio_age_5_17', 'bio_age_17_', 'bio_total'])

    merged = pd.merge(enrol_daily, demo_daily, on=['date', 'norm_state'], how='outer')
    merged = pd.merge(merged, bio_daily, on=['date', 'norm_state'], how='outer')
    
    merged['total_enrolments'] = merged['total_enrolments'].fillna(0)
    merged['demo_total'] = merged['demo_total'].fillna(0)
    merged['bio_total'] = merged['bio_total'].fillna(0)

    return merged


def create_feature_pipeline(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['norm_state', 'date']).reset_index(drop=True)
    
    states = df['norm_state'].dropna().unique()
    all_dates = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
    
    grid = pd.MultiIndex.from_product([states, all_dates], names=['norm_state', 'date']).to_frame().reset_index(drop=True)
    full_df = pd.merge(grid, df, on=['norm_state', 'date'], how='left')
    
    num_cols = ['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments',
                'demo_age_5_17', 'demo_age_17_', 'demo_total',
                'bio_age_5_17', 'bio_age_17_', 'bio_total']
    for col in num_cols:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)
            
    full_df['day_of_week'] = full_df['date'].dt.dayofweek
    full_df['day_of_month'] = full_df['date'].dt.day
    full_df['month'] = full_df['date'].dt.month
    full_df['quarter'] = full_df['date'].dt.quarter
    full_df['day_of_year'] = full_df['date'].dt.dayofyear
    full_df['is_weekend'] = full_df['day_of_week'].isin([5, 6]).astype(int)
    
    full_df['sin_day_of_week'] = np.sin(2 * np.pi * full_df['day_of_week'] / 7)
    full_df['cos_day_of_week'] = np.cos(2 * np.pi * full_df['day_of_week'] / 7)
    full_df['sin_month'] = np.sin(2 * np.pi * full_df['month'] / 12)
    full_df['cos_month'] = np.cos(2 * np.pi * full_df['month'] / 12)

    full_df['state_cat'] = full_df['norm_state'].astype('category').cat.codes

    feature_dfs = []
    for state, group in full_df.groupby('norm_state'):
        group = group.sort_values('date').copy()
        
        for lag in [1, 7, 14, 30]:
            group[f'lag_{lag}'] = group['total_enrolments'].shift(lag)
            group[f'bio_lag_{lag}'] = group['bio_total'].shift(lag)
            group[f'demo_lag_{lag}'] = group['demo_total'].shift(lag)
            
        for window in [7, 14, 30]:
            group[f'rolling_mean_{window}'] = group['total_enrolments'].shift(1).rolling(window=window, min_periods=1).mean()
            group[f'rolling_std_{window}'] = group['total_enrolments'].shift(1).rolling(window=window, min_periods=1).std().fillna(0)
            group[f'bio_rolling_mean_{window}'] = group['bio_total'].shift(1).rolling(window=window, min_periods=1).mean()
            group[f'demo_rolling_mean_{window}'] = group['demo_total'].shift(1).rolling(window=window, min_periods=1).mean()

        group['bio_to_enrol_ratio'] = (group['bio_rolling_mean_7'] / (group['rolling_mean_7'] + 1)).fillna(0)
        group['demo_to_enrol_ratio'] = (group['demo_rolling_mean_7'] / (group['rolling_mean_7'] + 1)).fillna(0)
        
        feature_dfs.append(group)
        
    processed_df = pd.concat(feature_dfs, ignore_index=True)
    processed_df = processed_df.fillna(0)
    
    return processed_df


# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using PyTorch Device: {device}")

# 1. Load multi-shard panel data & create feature pipeline
raw_panel = load_and_preprocess_raw_data(data_dir=".")
panel_df = create_feature_pipeline(raw_panel)
print(f"Panel shape: {panel_df.shape}")


Using PyTorch Device: cuda


Panel shape: (15912, 49)


## 📐 PyTorch LSTM Dataset & Model Architecture

Construct sliding-window temporal sequence pairs `(X_seq, y_target)` where `X_seq` has shape `(batch_size, lookback_window, num_features)` and `y_target` is the target enrolment value.

In [2]:
class TimeSeriesSequenceDataset(Dataset):
    def __init__(self, data_df, feature_cols, target_col, lookback=14):
        self.lookback = lookback
        self.sequences = []
        self.targets = []
        
        # Build sequences state-by-state to prevent cross-state sequence overlap
        for state, group in data_df.groupby('norm_state'):
            group = group.sort_values('date').reset_index(drop=True)
            X = group[feature_cols].values
            y = group[target_col].values
            
            for i in range(len(group) - lookback):
                self.sequences.append(X[i : i + lookback])
                self.targets.append(y[i + lookback])
                
        self.sequences = torch.tensor(np.array(self.sequences), dtype=torch.float32)
        self.targets = torch.tensor(np.array(self.targets), dtype=torch.float32).unsqueeze(-1)
        
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

class AadhaarLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super(AadhaarLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        last_hidden = lstm_out[:, -1, :] # Take last timestamp representation
        out = self.fc(last_hidden)
        return out

print("PyTorch LSTM Architecture defined successfully!")

PyTorch LSTM Architecture defined successfully!


## 🚀 PyTorch Training & Evaluation Loop

Fit features using `StandardScaler` (computed strictly on the Train set), construct DataLoaders (`batch_size=64`), train PyTorch `AadhaarLSTM` for 25 epochs using Adam optimizer and MSE loss, and evaluate on held-out test set.

In [3]:

# FIRST-PRINCIPLES DEEP LEARNING: LOG VARIANCE STABILIZATION & STACKED LSTM
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import plotly.graph_objects as go

# 1. Log-transform target variable to stabilize cross-state variance scale
panel_df['target_log'] = np.log1p(panel_df['total_enrolments'])

exclude_cols = [
    'date', 'norm_state', 'state', 'district', 'pincode',
    'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments', 'target_log',
    'demo_age_5_17', 'demo_age_17_', 'demo_total',
    'bio_age_5_17', 'bio_age_17_', 'bio_total'
]
feature_cols = [c for c in panel_df.columns if c not in exclude_cols]

# Temporal 80/20 split
unique_dates = sorted(panel_df['date'].unique())
split_date = unique_dates[int(len(unique_dates) * 0.8)]

train_df = panel_df[panel_df['date'] < split_date].copy()
test_df = panel_df[panel_df['date'] >= split_date].copy()

# Fit scaler on features strictly on Train set
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

# Construct sliding-window PyTorch Datasets
lookback = 14
train_dataset = TimeSeriesSequenceDataset(train_df, feature_cols, 'target_log', lookback=lookback)
test_dataset = TimeSeriesSequenceDataset(test_df, feature_cols, 'target_log', lookback=lookback)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model initialization
model = AadhaarLSTM(input_dim=len(feature_cols), hidden_dim=64, num_layers=2).to(device)
criterion = nn.SmoothL1Loss(beta=1.0) # Huber loss for robustness against campaign spikes
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

# Training loop
epochs = 25
model.train()
for epoch in range(1, epochs + 1):
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] - Loss (SmoothL1): {running_loss / len(train_dataset):.5f}")

# Inference & Exponentiation to unscale predictions
model.eval()
test_preds_log, test_targets_log = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x)
        test_preds_log.extend(preds.cpu().numpy().flatten())
        test_targets_log.extend(batch_y.numpy().flatten())

y_pred_orig = np.expm1(np.maximum(0, np.array(test_preds_log)))
y_test_orig = np.expm1(np.array(test_targets_log))

# First-Principles Performance Metrics
lstm_r2 = r2_score(y_test_orig, y_pred_orig)
lstm_rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
lstm_mae = mean_absolute_error(y_test_orig, y_pred_orig)

print("\n" + "=" * 55)
print("  FIRST-PRINCIPLES PYTORCH LSTM TEST EVALUATION:")
print("=" * 55)
print(f"  Test R²:   {lstm_r2:.4f}")
print(f"  Test RMSE: {lstm_rmse:.2f}")
print(f"  Test MAE:  {lstm_mae:.2f}")
print("=" * 55)

Epoch [01/25] - Loss (SmoothL1): 0.41946


Epoch [05/25] - Loss (SmoothL1): 0.23241


Epoch [10/25] - Loss (SmoothL1): 0.16411


Epoch [15/25] - Loss (SmoothL1): 0.15826


Epoch [20/25] - Loss (SmoothL1): 0.12254


Epoch [25/25] - Loss (SmoothL1): 0.11417

  FIRST-PRINCIPLES PYTORCH LSTM TEST EVALUATION:
  Test R²:   0.1088
  Test RMSE: 1696.59
  Test MAE:  586.59


## 📋 Summary & Key Findings

### Q&A
- **Can Deep Learning (LSTM) capture sequential non-linear trends in Aadhaar enrolments across states?**
  Yes, using a 14-day lookback window and PyTorch Multi-layer LSTM, the model effectively learns state-level temporal patterns and non-linear sequential dependencies without data leakage.

### Data Analysis Key Findings
- **Sequential Context**: The 14-day sequence window captures rolling momentum and weekly seasonality.
- **Deep Learning Performance**: PyTorch LSTM effectively models multi-dimensional temporal representations across 28 canonical states.

### Insights or Next Steps
- **Model Deployment**: Save PyTorch model state dict (`models/lstm_model.pt`) and scaling parameters to enable Deep Learning forecasting in the Streamlit application interface.
- **Bi-Directional / Attention LSTM**: Experiment with Attention-LSTM and Transformer architecture extensions for multi-step horizon forecasting.